In [0]:
from datetime import datetime
import uuid

batch_id = f"MIGRATION_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_id = str(uuid.uuid4())

print("Batch ID :", batch_id)
print("Run ID   :", run_id)

# Connection String

In [0]:
jdbc_url = "jdbc:postgresql://0.tcp.in.ngrok.io:29031/demo"

username = "postgres"
password = "root"

driver = "org.postgresql.Driver"

In [0]:
from datetime import datetime
import uuid

batch_id = f"MIGRATION_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_id = str(uuid.uuid4())

print("Batch ID :", batch_id)
print("Run ID   :", run_id)

In [0]:
customers_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "public.customers")
    .option("user", username)
    .option("password", password)
    .option("driver", driver)
    .load()
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_customers_df = (
    customers_df
    .withColumn("_batch_id", lit(batch_id))
    .withColumn("_run_id", lit(run_id))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_system", lit("PostgreSQL"))
    .withColumn("_source_table", lit("customers"))
)

display(bronze_customers_df)

In [0]:
(
    bronze_customers_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("migration.bronze.customers")
)

In [0]:
%sql
SELECT *
FROM migration.bronze.customers
ORDER BY customer_id;

# Source Watermark

In [0]:
source_watermark = (
    customers_df
    .agg({"updated_at": "max"})
    .collect()[0][0]
)

print("Source watermark:", source_watermark)

In [0]:
row_count = customers_df.count()

print("Row count:", row_count)

# create a one-row DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp

control_df = spark.createDataFrame(
    [
        (
            "PostgreSQL",
            "customers",
            source_watermark,
            batch_id,
            run_id,
            "SUCCESS",
            row_count
        )
    ],
    """
    source_system STRING,
    source_table STRING,
    last_watermark TIMESTAMP,
    last_batch_id STRING,
    last_run_id STRING,
    last_status STRING,
    last_row_count BIGINT
    """
).withColumn(
    "last_processed_at",
    current_timestamp()
)

# Create the control table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS migration.control.ingestion_state (
    source_system STRING,
    source_table STRING,
    last_watermark TIMESTAMP,
    last_batch_id STRING,
    last_run_id STRING,
    last_status STRING,
    last_processed_at TIMESTAMP,
    last_row_count BIGINT
)
USING DELTA;

# one-row DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp

control_df = spark.createDataFrame(
    [
        (
            "PostgreSQL",
            "customers",
            source_watermark,
            batch_id,
            run_id,
            "SUCCESS",
            row_count
        )
    ],
    """
    source_system STRING,
    source_table STRING,
    last_watermark TIMESTAMP,
    last_batch_id STRING,
    last_run_id STRING,
    last_status STRING,
    last_row_count BIGINT
    """
).withColumn(
    "last_processed_at",
    current_timestamp()
)

In [0]:
display(control_df)

In [0]:
(
    control_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("migration.control.ingestion_state")
)

In [0]:
%sql
SELECT
    source_system,
    source_table,
    last_watermark,
    last_batch_id,
    last_run_id,
    last_status,
    last_processed_at,
    last_row_count
FROM migration.control.ingestion_state;